In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_bureau")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-55772b53-3b33-44cb-83e7-6fccb712d0e1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (714ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (75ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (104ms)
:: resolution report :: resolve 1922ms :: artifacts dl 8

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://bronze/base_score_bureau_movel/"
df_base_score_bureau_movel = spark.read.parquet(path)
df_base_score_bureau_movel.show(5, truncate=False)

25/12/30 07:31:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+------+---------------+---+----+---------+--------+--------+-----------+
|SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|NUM_CPF    |
+------+---------------+---+----+---------+--------+--------+-----------+
|202410|1              |0  |CMV |PRE      |562     |636     |ZZZZZX7XWY8|
|202410|1              |1  |CMV |PRE      |546     |518     |ZZZZZX88YXY|
|202410|1              |0  |CMV |PRE      |621     |750     |ZZZZZYT7XYT|
|202410|1              |1  |CMV |PRE      |609     |679     |ZZZZZNTXY9Z|
|202410|1              |0  |CMV |PRE      |621     |722     |ZZZZZ79ZXUX|
+------+---------------+---+----+---------+--------+--------+-----------+
only showing top 5 rows



In [4]:
df_base_score_bureau_movel.createOrReplaceTempView("raw_00")

In [5]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM raw_00
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5, truncate=False)

+------+------------+-------------+
|SAFRA |total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|203828      |203828       |
|202411|227176      |227176       |
|202412|227985      |227985       |
|202501|221002      |221002       |
|202502|203139      |203139       |
+------+------------+-------------+
only showing top 5 rows



In [6]:
print('lista de colunas para tipar')
for col in spark.table("raw_00").columns:
    print('cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
cast(SAFRA as) as SAFRA,
cast(FLAG_INSTALACAO as) as FLAG_INSTALACAO,
cast(FPD as) as FPD,
cast(PROD as) as PROD,
cast(flag_mig2 as) as flag_mig2,
cast(SCORE_01 as) as SCORE_01,
cast(SCORE_02 as) as SCORE_02,
cast(NUM_CPF as) as NUM_CPF,


In [7]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            cast(NUM_CPF as string) as NUM_CPF,
            cast(SAFRA as int) as SAFRA,
            cast(FLAG_INSTALACAO as int) as FLAG_INSTALACAO,
            cast(FPD as int) as FPD,
            cast(PROD as string) as PROD,
            cast(flag_mig2 as string) as flag_mig2,
            cast(SCORE_01 as int) as SCORE_01,
            cast(SCORE_02 as int) as SCORE_02,
            {pdthproc} as DATPROC

        from
            raw_00
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

1290526

In [8]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- FLAG_INSTALACAO: integer (nullable = true)
 |-- FPD: integer (nullable = true)
 |-- PROD: string (nullable = true)
 |-- flag_mig2: string (nullable = true)
 |-- SCORE_01: integer (nullable = true)
 |-- SCORE_02: integer (nullable = true)
 |-- DATPROC: long (nullable = false)



In [10]:
lake.show(5, truncate=False)

+-----------+------+---------------+---+----+---------+--------+--------+--------------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|DATPROC       |
+-----------+------+---------------+---+----+---------+--------+--------+--------------+
|ZZZZZX7XWY8|202410|1              |0  |CMV |PRE      |562     |636     |20251230073052|
|ZZZZZX88YXY|202410|1              |1  |CMV |PRE      |546     |518     |20251230073052|
|ZZZZZYT7XYT|202410|1              |0  |CMV |PRE      |621     |750     |20251230073052|
|ZZZZZNTXY9Z|202410|1              |1  |CMV |PRE      |609     |679     |20251230073052|
|ZZZZZ79ZXUX|202410|1              |0  |CMV |PRE      |621     |722     |20251230073052|
+-----------+------+---------------+---+----+---------+--------+--------+--------------+
only showing top 5 rows



In [11]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, SAFRA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

1290526

In [12]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(5)

+------+------------+-------------+
| SAFRA|total_linhas|cpf_distintos|
+------+------------+-------------+
|202410|      203828|       203828|
|202411|      227176|       227176|
|202412|      227985|       227985|
|202501|      221002|       221002|
|202502|      203139|       203139|
+------+------------+-------------+
only showing top 5 rows



In [13]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_score_bureau_movel/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [14]:
spark.stop()